# Data cleaning: stores, products, transactions

(notebook generated via LLM, Prompt #3 in the llm_transcripts.txt file )

This notebook turns the three raw files into validated tables. Every change is a decision that was made after profiling the data. Each one is logged below, along with the reason it was made.

**Inputs:** `../data/raw/stores.csv`, `products.csv`, `transactions.csv` (never modified)
**Outputs:** `../data/validated/`
- `stores.csv`, `products.csv`, `transactions.csv`: the cleaned tables
- `transactions_dropped_zero_quantity.csv`: the zero-quantity transactions that were dropped, for the client
- `cleaning_log.csv`: the decision log with the counts measured on this run

**Rerunning:** use *Kernel → Restart & Run All*, or run it headless from this folder:
```
jupyter nbconvert --to notebook --execute --inplace data_cleaning.ipynb
```
Each section starts again from the raw data, so the notebook gives the same result on every run and the outputs are overwritten.

**Safety checks:** every step checks the record count it expects, using the counts found during profiling. If the raw data changes and a count no longer matches, the notebook stops at that step (`STRICT = True`) so someone can review it before anything is written. Set `STRICT = False` in the config cell to turn those stops into warnings.

## Decision log

These are the decisions and reasons given during the data review. The counts are the ones found during profiling. The last section of the notebook rebuilds this table using the counts measured on this run.

| Issue | File | Count | Decision | Rationale |
|---|---|---|---|---|
| `region` is blank for both stores with `state` = 'OR' | stores | 2 | Fill `region` with "West" | Based on the other state/region columns, we can impute the region "West" here. |
| `S007` appears twice with different `store_name`s | stores | 2 rows (1 dropped) | Drop one of the two rows | It's the same record. |
| Invalid zip codes | stores | 2 | Keep `zip_code` as text. Flag zips that are not 5 digits in a new column `zip_code_invalid`. Do not impute | Text avoids truncating leading/trailing zeros. It's too risky to impute new zips without being sure, but if zip code isn't necessary for analytics, it can be safe to keep in the data for now. |
| Products with no `category` | products | 5 | Fill from the category of other products with the same `supplier_id` | Each supplier maps to exactly one category. |
| Exact duplicate rows | products | 1 | Drop | exact duplicates can be safely dropped |
| `P005` appears twice with different prices | products | 2 rows (1 dropped) | Keep the row whose price matches `unit_price` in transactions | Only one matches the price in the transactions table. |
| `P027` has a catalog price of 0.00 | products | 1 | Impute the price from the transactions `unit_price` | It has a price in the transactions table. |
| Exact duplicate rows | transactions | 15 | Deduplicate | exact duplicates can be safely dropped |
| `quantity` and `total_amount` are both 0 | transactions | 5 | Drop, and flag for the client | We drop them so they don't skew average transaction value or revenue tracking. |
| Negative values are returns that match earlier sales | transactions | 30 negatives | Flag each return and its matching sale. Do not drop | So they aren't counted in revenue or other metrics, but aren't dropped, so we can still track them as needed. |
| `$` in `total_amount` | transactions | 25 with `$` | Remove the `$`, then cast to a decimal number | — the $ can be stripped safely so that the column can be cast as numberic |
| Mixed date formats where day and month are ambiguous (both ≤ 12) | transactions | 7 values | Cast to YYYY-MM-DD, resolving each ambiguous value as below | The dates range from March to June, so: `01-04-2026` (TXN10011) is April 1st. `04/12/2026` and `04/01/2026` are April. `05/07/2026` is May. `12-03-2026` is March. `05/05/2026` can go either way, so it doesn't matter. That leaves one record, `06-04-2026`, which is June 4th, 2026. |
| Stores in transactions but not in the stores file (S016–S019) | stores | 4 stores (5 transactions) | Add placeholder stores with details `Unknown` and `store_details_missing` = True. Keep the sales. Flag the stores for the client | Instead of dropping these records and losing data, we can just add the stores to the stores table and keep the transactions.|

## 1. Setup

All decisions that depend on the data are in the config cell below: the expected counts, the region fill, the date window and the date resolutions. If a decision changes, edit it here.

In [1]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# ---- Paths (relative to this notebook) ----
RAW_DIR = Path("../data/raw")
OUT_DIR = Path("../data/validated")
RAW_FILES = {"stores": "stores.csv", "products": "products.csv", "transactions": "transactions.csv"}

# ---- Behaviour ----
STRICT = True  # True: stop when a check fails. False: print a warning and carry on.

EXPECTED_COLUMNS = {
    "stores": ["store_id", "store_name", "city", "state", "zip_code", "region", "opened_date"],
    "products": ["product_id", "product_name", "category", "unit_price", "supplier_id"],
    "transactions": ["transaction_id", "transaction_date", "store_id", "product_id",
                     "customer_id", "quantity", "unit_price", "total_amount"],
}

# ---- Counts found during profiling (each step checks against these) ----
EXPECTED = {
    "stores_blank_region_to_fill": 2,       # state = OR, region blank
    "stores_s007_rows": 2,
    "stores_invalid_zip": 1,
    "products_blank_category": 5,
    "products_p005_rows": 2,
    "products_zero_price": 1,               # P027
    "transactions_exact_duplicates": 15,
    "transactions_zero_qty_and_total": 5,
}

# ---- Decisions ----
REGION_FILL_BY_STATE = {"OR": "West"}
DUPLICATE_STORE_ID = "S007"   # same record twice; the first row in file order is kept
PRICE_FROM_TRANSACTIONS = {"duplicate": "P005", "zero_price": "P027"}

DATE_WINDOW = ("2026-03-01", "2026-06-30")  # transactions run from March to June 2026
# Day and month are both <= 12 in these values, so each one is resolved by decision:
AMBIGUOUS_DATE_RESOLUTIONS = {
    "01-04-2026": "2026-04-01",  # TXN10011: April 1st (dates range March-June)
    "04/12/2026": "2026-04-12",  # April
    "04/01/2026": "2026-04-01",  # April
    "05/07/2026": "2026-05-07",  # May
    "12-03-2026": "2026-03-12",  # March
    "05/05/2026": "2026-05-05",  # same either way
    "06-04-2026": "2026-06-04",  # the one record left: June 4th, 2026
}

In [2]:
# ---- Helpers ----
LOG = []  # one entry per decision, with the counts measured on this run


def check(ok, message):
    """Print a pass, or stop (STRICT) / warn when the data doesn't look as expected."""
    if ok:
        print(f"PASS  {message}")
        return True
    if STRICT:
        raise AssertionError(f"CHECK FAILED: {message}\n"
                             "STRICT = True, so the notebook stops here for review. "
                             "Set STRICT = False in the config cell to continue with a warning.")
    print(f"WARN  {message}")
    return False


def expect(key, actual, what):
    return check(actual == EXPECTED[key], f"{what}: expected {EXPECTED[key]}, found {actual}")


def info(message):
    print(f"INFO  {message}")


def record(issue, file, count, decision, rationale):
    LOG.append({"Issue": issue, "File": file, "Count": count, "Decision": decision, "Rationale": rationale})


def is_blank(s):
    """True for missing values and empty or whitespace-only strings."""
    return s.isna() | s.astype(str).str.strip().eq("")


def to_number(s, label):
    """Cast text to a number. Stops on any non-blank value that isn't a plain number."""
    out = pd.to_numeric(s.astype(str).str.strip().where(~is_blank(s)), errors="coerce")
    bad = out.isna() & ~is_blank(s)
    if bad.any():
        raise ValueError(f"{label}: {bad.sum()} values are not numbers, e.g. {s[bad].unique()[:5].tolist()}")
    return out


def to_markdown(df):
    """Render a small DataFrame as a markdown table (no extra packages needed)."""
    esc = lambda v: str(v).replace("|", "\\|").replace("\n", " ")
    lines = ["| " + " | ".join(df.columns) + " |", "|" + "---|" * len(df.columns)]
    lines += ["| " + " | ".join(esc(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(lines)

### Load the raw data

Every column is read as text (`dtype=str`, with no automatic NA conversion). This keeps the raw values as they are, so zip codes keep their leading zeros and values like `$10.00` can be seen and handled on purpose. Each column is converted to its proper type in the step that cleans it.

In [3]:
if not RAW_DIR.is_dir():
    raise FileNotFoundError(f"Raw data folder not found: {RAW_DIR.resolve()}")

raw = {name: pd.read_csv(RAW_DIR / fname, dtype=str, keep_default_na=False)
       for name, fname in RAW_FILES.items()}

for name, df in raw.items():
    print(f"{name:13s} {df.shape[0]:>6,} rows x {df.shape[1]} columns")
    check(list(df.columns) == EXPECTED_COLUMNS[name], f"{name} has the expected columns")

stores            16 rows x 7 columns
PASS  stores has the expected columns
products          32 rows x 5 columns
PASS  products has the expected columns
transactions     505 rows x 8 columns
PASS  transactions has the expected columns


## 2. Stores

### 2.1 Fill the blank region for OR stores
`region` is blank for both stores with `state` = 'OR'. Based on the other state/region columns, we can impute the region "West" here. The table below shows the states in each region, from the rows where `region` is filled in, as the basis for that decision.

In [4]:
stores = raw["stores"].copy()

display(stores[~is_blank(stores["region"])].groupby("region")["state"].unique().to_frame("states"))

to_fill = is_blank(stores["region"]) & stores["state"].isin(REGION_FILL_BY_STATE)
display(stores[to_fill])
expect("stores_blank_region_to_fill", int(to_fill.sum()), "stores with state OR and a blank region")

stores.loc[to_fill, "region"] = stores.loc[to_fill, "state"].map(REGION_FILL_BY_STATE)

check(not is_blank(stores["region"]).any(), "no blank regions remain")
record("`region` blank for both stores with state 'OR'", "stores", int(to_fill.sum()),
       'Filled `region` with "West"',
       'Based on the other state/region columns, we can impute the region "West" here.')

,states
region,
Midwest,[MN]
Northeast,[NY]
South,"[LA, TX]"
West,"[AZ, WA]"


,store_id,store_name,city,state,zip_code,region,opened_date
13,S013,Cascade Station,Portland,OR,97220,,2019-05-14
14,S014,Lloyd Center,Portland,OR,97232,,2007-09-09


PASS  stores with state OR and a blank region: expected 2, found 2
PASS  no blank regions remain


### 2.2 Remove the duplicate S007
`S007` appears twice with different `store_name`s. It's the same record, so one row is dropped. The first row in file order is kept, and both rows are shown below for reference.

In [5]:
s007 = stores[stores["store_id"] == DUPLICATE_STORE_ID]
display(s007)
expect("stores_s007_rows", len(s007), f"rows for {DUPLICATE_STORE_ID}")

other_cols = [c for c in stores.columns if c != "store_name"]
check(len(s007[other_cols].drop_duplicates()) == 1,
      f"the {DUPLICATE_STORE_ID} rows differ only in store_name")

dropped = s007.index[1:]
stores = stores.drop(index=dropped)

check(stores["store_id"].is_unique, "store_id is unique")
record(f"`{DUPLICATE_STORE_ID}` appears twice with different store_names", "stores",
       f"{len(s007)} rows ({len(dropped)} dropped)",
       "Dropped one row (kept the first in file order)", "It's the same record.")

,store_id,store_name,city,state,zip_code,region,opened_date
6,S007,Downtown Rochester,Rochester,NY,14604,Northeast,2006-01-22
7,S007,Rochester Downtown,Rochester,NY,14604,Northeast,2006-01-22


PASS  rows for S007: expected 2, found 2
PASS  the S007 rows differ only in store_name
PASS  store_id is unique


### 2.3 Flag invalid zip codes
`zip_code` stays as text, so leading or trailing zeros can't be cut off. Any zip that still isn't exactly 5 digits is flagged in a new column, `zip_code_invalid`, for the client. The values are not changed: it's too risky to impute new zips without being sure, but if zip code isn't needed for analytics, it can be safe to keep in the data for now.

In [6]:
stores["zip_code"] = stores["zip_code"].astype("string")
stores["zip_code_invalid"] = ~stores["zip_code"].str.fullmatch(r"\d{5}").fillna(False).astype(bool)

display(stores[stores["zip_code_invalid"]])
expect("stores_invalid_zip", int(stores["zip_code_invalid"].sum()), "zip codes that are not 5 digits")

record("Invalid zip codes", "stores", int(stores["zip_code_invalid"].sum()),
       "Kept `zip_code` as text; flagged zips that are not 5 digits in `zip_code_invalid`; not imputed",
       "Text avoids truncating leading/trailing zeros. It's too risky to impute new zips without being "
       "sure, but if zip code isn't necessary for analytics, it can be safe to keep in the data for now.")

,store_id,store_name,city,state,zip_code,region,opened_date,zip_code_invalid
2,S003,Greece Ridge Center,Greece,NY,0938,Northeast,2012-06-20,True


PASS  zip codes that are not 5 digits: expected 1, found 1


## 3. Products

The transactions table is the source for two of the product price decisions, so its `unit_price` values are read here from the raw file.

In [7]:
products = raw["products"].copy()
txn_prices = raw["transactions"][["product_id"]].assign(
    unit_price=to_number(raw["transactions"]["unit_price"], "transactions.unit_price").round(2))

### 3.1 Fill missing categories from the supplier
5 products have no `category`. Each supplier maps to exactly one category, so the blanks are filled using the category of the other rows from the same `supplier_id`. The first check below confirms that each supplier maps to exactly one category.

In [8]:
blank_cat = is_blank(products["category"])
populated = products[~blank_cat]

cats_per_supplier = populated.groupby("supplier_id")["category"].nunique()
check((cats_per_supplier == 1).all(),
      f"every supplier maps to exactly one category ({len(cats_per_supplier)} suppliers)")
supplier_category = populated.drop_duplicates("supplier_id").set_index("supplier_id")["category"]

display(products[blank_cat])
expect("products_blank_category", int(blank_cat.sum()), "products with no category")

fill = products.loc[blank_cat, "supplier_id"].map(supplier_category)
check(fill.notna().all(), "every product with a blank category has a supplier with a known category")
products.loc[blank_cat, "category"] = fill

display(products[blank_cat])
record("Products with no `category`", "products", int(blank_cat.sum()),
       "Filled from the category of other products with the same `supplier_id`",
       "Each supplier maps to exactly one category.")

PASS  every supplier maps to exactly one category (5 suppliers)


,product_id,product_name,category,unit_price,supplier_id
1,P016,Product P016,,24.52,SUP001
4,P009,Product P009,,67.73,SUP004
17,P003,Product P003,,134.2,SUP003
22,P023,Product P023,,108.28,SUP003
29,P029,Product P029,,155.01,SUP004


PASS  products with no category: expected 5, found 5
PASS  every product with a blank category has a supplier with a known category


,product_id,product_name,category,unit_price,supplier_id
1,P016,Product P016,Electronics,24.52,SUP001
4,P009,Product P009,Food & Beverage,67.73,SUP004
17,P003,Product P003,Home & Garden,134.2,SUP003
22,P023,Product P023,Home & Garden,108.28,SUP003
29,P029,Product P029,Food & Beverage,155.01,SUP004


### 3.2 Remove duplicate rows, and keep the right P005
All exact duplicate rows are dropped. `P005` appears twice with different prices, and only one of them matches the price in the transactions table, so that's the row that's kept.

In [9]:
n_exact = int(products.duplicated().sum())
display(products[products.duplicated(keep=False)])
products = products.drop_duplicates()
info(f"{n_exact} exact duplicate product rows dropped")
record("Exact duplicate rows", "products", n_exact, "Dropped", "— (no rationale given)")

products["unit_price"] = to_number(products["unit_price"], "products.unit_price").round(2)

pid = PRICE_FROM_TRANSACTIONS["duplicate"]
rows = products[products["product_id"] == pid]
txn_price_set = set(txn_prices.loc[txn_prices["product_id"] == pid, "unit_price"].dropna())
display(rows)
info(f"{pid} unit_price in transactions: {sorted(float(p) for p in txn_price_set)}")
expect("products_p005_rows", len(rows), f"rows for {pid}")

matches = rows["unit_price"].isin(txn_price_set)
check(int(matches.sum()) == 1, f"exactly one {pid} row matches a transaction price")
products = products.drop(index=rows.index[~matches.to_numpy()])

check(products["product_id"].is_unique, "product_id is unique")
record(f"`{pid}` appears twice with different prices", "products",
       f"{len(rows)} rows ({int((~matches).sum())} dropped)",
       "Kept the row whose price matches the transactions `unit_price`",
       "Only one matches the price in the transactions table.")

,product_id,product_name,category,unit_price,supplier_id
14,P012,Product P012,Apparel,123.17,SUP002
24,P012,Product P012,Apparel,123.17,SUP002


INFO  1 exact duplicate product rows dropped


,product_id,product_name,category,unit_price,supplier_id
6,P005,Product P005,Office Supplies,150.11,SUP005
10,P005,Product P005,Office Supplies,141.61,SUP005


INFO  P005 unit_price in transactions: [141.61]
PASS  rows for P005: expected 2, found 2
PASS  exactly one P005 row matches a transaction price
PASS  product_id is unique


### 3.3 Impute P027's price from transactions
`P027` has a catalog price of 0.00, but it has a price in the transactions table, so that `unit_price` is used.

In [10]:
pid = PRICE_FROM_TRANSACTIONS["zero_price"]
zero_price = products["unit_price"].eq(0)
display(products[zero_price])
expect("products_zero_price", int(zero_price.sum()), "products with a price of 0.00")
check(products.loc[zero_price, "product_id"].tolist() == [pid], f"the zero-price product is {pid}")

prices = sorted(float(p) for p in txn_prices.loc[txn_prices["product_id"] == pid, "unit_price"].dropna().unique())
info(f"{pid} unit_price in transactions: {prices}")
check(len(prices) == 1, f"{pid} has exactly one distinct unit_price in transactions")

products.loc[products["product_id"] == pid, "unit_price"] = prices[0]
display(products[products["product_id"] == pid])
record(f"`{pid}` has a catalog price of 0.00", "products", int(zero_price.sum()),
       f"Imputed the price from the transactions `unit_price` ({prices[0]:.2f})",
       "It has a price in the transactions table.")

,product_id,product_name,category,unit_price,supplier_id
18,P027,Product P027,Apparel,0.0,SUP002


PASS  products with a price of 0.00: expected 1, found 1
PASS  the zero-price product is P027
INFO  P027 unit_price in transactions: [195.34]
PASS  P027 has exactly one distinct unit_price in transactions


,product_id,product_name,category,unit_price,supplier_id
18,P027,Product P027,Apparel,195.34,SUP002


## 4. Transactions

The steps run in this order: deduplicate → clean amounts → drop zero rows → fix dates → flag returns. The amounts have to be numbers before zero rows can be found, and the dates have to be standardized before returns can be matched to sales on date.

### 4.1 Remove exact duplicate rows
15 rows are exact duplicates, and they're removed. This runs on the raw text, before any other change.

In [11]:
transactions = raw["transactions"].copy()

n_dup = int(transactions.duplicated().sum())
expect("transactions_exact_duplicates", n_dup, "exact duplicate transaction rows")
transactions = transactions.drop_duplicates().reset_index(drop=True)

record("Exact duplicate rows", "transactions", n_dup, "Deduplicated", "— (no rationale given)")
info(f"{len(transactions):,} rows remain")
n_id_dups = int(transactions["transaction_id"].duplicated().sum())
info(f"{n_id_dups} transaction_ids still appear more than once" if n_id_dups else "transaction_id is unique")

PASS  exact duplicate transaction rows: expected 15, found 15
INFO  490 rows remain
INFO  transaction_id is unique


### 4.2 Remove `$` from `total_amount` and cast it to a decimal
Any `$` in `total_amount` is removed, and the column is cast to a decimal number. `quantity` and `unit_price` are cast to numbers too, which the later steps need. Those casts stop on any value that isn't a plain number, so nothing gets coerced without anyone seeing it.

In [12]:
has_dollar = transactions["total_amount"].str.contains("$", regex=False)
display(transactions[has_dollar].head())
info(f"{int(has_dollar.sum())} total_amount values contain '$'")

transactions["total_amount"] = to_number(
    transactions["total_amount"].str.replace("$", "", regex=False), "total_amount").round(2)
transactions["unit_price"] = to_number(transactions["unit_price"], "unit_price").round(2)
quantity = to_number(transactions["quantity"], "quantity")
check(quantity.notna().all() and (quantity % 1 == 0).all(), "quantity is a whole number on every row")
transactions["quantity"] = quantity.astype("int64")

check(not transactions["total_amount"].isna().any(), "total_amount is populated on every row")
record("`$` in `total_amount`", "transactions", int(has_dollar.sum()),
       "Removed `$`, cast `total_amount` to a decimal number", "— (no rationale given)")

mismatch = (transactions["quantity"] * transactions["unit_price"]).round(2) != transactions["total_amount"]
info(f"{int(mismatch.sum())} rows where quantity x unit_price != total_amount (not changed)")

,transaction_id,transaction_date,store_id,product_id,customer_id,quantity,unit_price,total_amount
12,TXN10041,2026-05-14,S014,P012,CUST0043,5,123.17,$615.85
19,TXN10033,2026-04-01,S005,P003,CUST0069,1,134.2,$134.20
21,TXN10038,2026-03-26,S014,P010,CUST0142,5,237.08,$1185.40
33,TXN10031,2026-05-19,S008,P017,CUST0064,3,126.96,$380.88
39,TXN10032,2026-03-29,S001,P017,CUST0111,4,126.96,$507.84


INFO  25 total_amount values contain '$'
PASS  quantity is a whole number on every row
PASS  total_amount is populated on every row
INFO  20 rows where quantity x unit_price != total_amount (not changed)


### 4.3 Drop transactions with zero quantity and total, and flag them for the client
5 transactions have a `quantity` and `total_amount` of 0. They're dropped so they don't skew average transaction value or revenue tracking. They're also saved to `transactions_dropped_zero_quantity.csv` so the client can see them.

In [13]:
zero = transactions["quantity"].eq(0) & transactions["total_amount"].eq(0)
display(transactions[zero])
expect("transactions_zero_qty_and_total", int(zero.sum()), "transactions with quantity and total_amount of 0")

only_one_zero = transactions["quantity"].eq(0) ^ transactions["total_amount"].eq(0)
info(f"{int(only_one_zero.sum())} rows where only one of quantity / total_amount is 0 (kept)")

zero_dropped = transactions[zero].assign(
    flag_reason="quantity and total_amount are both 0; dropped from validated transactions")
transactions = transactions[~zero].reset_index(drop=True)

record("`quantity` and `total_amount` both 0", "transactions", int(zero.sum()),
       "Dropped; saved to `transactions_dropped_zero_quantity.csv` for the client",
       "We drop them so they don't skew average transaction value or revenue tracking.")

,transaction_id,transaction_date,store_id,product_id,customer_id,quantity,unit_price,total_amount
43,TXN10251,2026-04-07,S001,P025,CUST0037,0,140.26,0.0
279,TXN10253,2026-05-19,S007,P001,CUST0236,0,157.39,0.0
323,TXN10252,2026-04-23,S002,P022,CUST0249,0,216.88,0.0
378,TXN10254,2026-03-19,S004,P022,CUST0272,0,216.88,0.0
388,TXN10255,2026-05-14,S007,P004,CUST0181,0,10.53,0.0


PASS  transactions with quantity and total_amount of 0: expected 5, found 5
INFO  0 rows where only one of quantity / total_amount is 0 (kept)


### 4.4 Standardize dates to YYYY-MM-DD
Each `transaction_date` is parsed as follows:
- `YYYY-MM-DD` is taken as is.
- `NN/NN/YYYY` or `NN-NN-YYYY` where one number is above 12: that number has to be the day.
- Day and month the same (`05/05/2026`): it reads the same either way.
- Both numbers ≤ 12 and different: resolved only from `AMBIGUOUS_DATE_RESOLUTIONS` in the config cell (the decisions in the log). An ambiguous value that isn't listed there stops the notebook. It is never guessed.

Every parsed date is then checked to be between March and June 2026.

In [14]:
def parse_transaction_date(value):
    """Return (Timestamp or None, how it was parsed)."""
    v = value.strip()
    if v in AMBIGUOUS_DATE_RESOLUTIONS:
        return pd.Timestamp(AMBIGUOUS_DATE_RESOLUTIONS[v]), "ambiguous: resolved by decision"
    try:
        if m := re.fullmatch(r"(\d{4})-(\d{1,2})-(\d{1,2})", v):
            y, mo, d = map(int, m.groups())
            return pd.Timestamp(year=y, month=mo, day=d), "YYYY-MM-DD"
        if m := re.fullmatch(r"(\d{1,2})([/-])(\d{1,2})\2(\d{4})", v):
            a, sep, b, y = int(m[1]), m[2], int(m[3]), int(m[4])
            if a > 12 >= b:
                return pd.Timestamp(year=y, month=b, day=a), f"DD{sep}MM{sep}YYYY"
            if b > 12 >= a:
                return pd.Timestamp(year=y, month=a, day=b), f"MM{sep}DD{sep}YYYY"
            if a == b:
                return pd.Timestamp(year=y, month=a, day=b), "day = month"
            return None, "ambiguous: NOT resolved"
    except ValueError:
        return None, "invalid date"
    return None, "unrecognised format"


parsed = {v: parse_transaction_date(v) for v in transactions["transaction_date"].unique()}
how = transactions["transaction_date"].map(lambda v: parsed[v][1])
display(how.value_counts().to_frame("rows"))

unresolved = transactions[transactions["transaction_date"].map(lambda v: parsed[v][0] is None)]
if len(unresolved):
    display(unresolved.assign(reason=how[unresolved.index]))
check(unresolved.empty, "every transaction_date was parsed")

ambiguous = transactions[how.eq("ambiguous: resolved by decision")]
display(ambiguous[["transaction_id", "transaction_date"]].assign(
    resolved_to=ambiguous["transaction_date"].map(AMBIGUOUS_DATE_RESOLUTIONS)))
unused = set(AMBIGUOUS_DATE_RESOLUTIONS) - set(transactions["transaction_date"].str.strip())
if unused:
    info(f"resolutions in the config that don't appear in the data: {sorted(unused)}")

transactions["transaction_date"] = transactions["transaction_date"].map(lambda v: parsed[v][0])
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"])

txn10011 = transactions.loc[transactions["transaction_id"] == "TXN10011", "transaction_date"]
check(txn10011.eq(pd.Timestamp("2026-04-01")).all() and len(txn10011) > 0, "TXN10011 is dated 2026-04-01")
lo, hi = map(pd.Timestamp, DATE_WINDOW)
check(transactions["transaction_date"].between(lo, hi).all(),
      f"all dates fall between {DATE_WINDOW[0]} and {DATE_WINDOW[1]} "
      f"(actual {transactions['transaction_date'].min().date()} to {transactions['transaction_date'].max().date()})")

record("Mixed date formats; day/month ambiguous where both <= 12", "transactions",
       f"{transactions['transaction_date'].size} dates cast; {len(ambiguous)} rows "
       f"({ambiguous['transaction_date'].nunique()} values) resolved by decision",
       "Cast to YYYY-MM-DD; ambiguous values resolved as listed in the config",
       "The dates range from March to June: 01-04-2026 (TXN10011) is April 1st; 04/12/2026 and 04/01/2026 are "
       "April; 05/07/2026 is May; 12-03-2026 is March; 05/05/2026 can go either way, so it doesn't matter. "
       "That leaves one record, 06-04-2026, which is June 4th, 2026.")

,rows
transaction_date,
YYYY-MM-DD,465
ambiguous: resolved by decision,7
DD-MM-YYYY,7
MM/DD/YYYY,6


PASS  every transaction_date was parsed


,transaction_id,transaction_date,resolved_to
49,TXN10011,01-04-2026,2026-04-01
132,TXN10001,04/12/2026,2026-04-12
163,TXN10009,04/01/2026,2026-04-01
193,TXN10002,05/07/2026,2026-05-07
203,TXN10020,12-03-2026,2026-03-12
288,TXN10006,05/05/2026,2026-05-05
430,TXN10017,06-04-2026,2026-06-04


PASS  TXN10011 is dated 2026-04-01
PASS  all dates fall between 2026-03-01 and 2026-06-30 (actual 2026-03-05 to 2026-06-27)


### 4.5 Flag returns and the sales they reverse
All negative values are returns, and each matches an earlier sale with the same `store_id`, `product_id`, `customer_id` and `transaction_date`. Both the return and its matching sale are flagged so they aren't counted in revenue or other metrics. They aren't dropped, so they can still be tracked as needed.

New columns:
- `is_return`: the row is a return (negative quantity)
- `is_returned_sale`: the row is the sale a return reverses
- `matched_transaction_id`: the transaction on the other side of the pair
- `exclude_from_metrics`: `is_return OR is_returned_sale`. Filter on this for revenue and other metrics.

Each return is matched to one sale. If there are several candidates, the sale with the same quantity is preferred, then the earliest `transaction_id`.

In [15]:
EXPECTED.setdefault("stores_missing_from_file", 4)

KEYS = ["store_id", "product_id", "customer_id", "transaction_date"]
is_return = transactions["quantity"] < 0
check(((transactions["total_amount"] < 0) == is_return).all(),
      "negative total_amount occurs on exactly the rows with negative quantity")

sales = transactions[~is_return].sort_values("transaction_id")
candidates = {k: list(g.index) for k, g in sales.groupby(KEYS, sort=False)}

matched_sale = {}  # return index -> sale index
used = set()
for r_idx, r in transactions[is_return].sort_values("transaction_id").iterrows():
    options = [i for i in candidates.get(tuple(r[KEYS]), []) if i not in used
               and transactions.at[i, "quantity"] >= -r["quantity"]]
    if options:
        options.sort(key=lambda i: transactions.at[i, "quantity"] != -r["quantity"])  # exact qty first
        matched_sale[r_idx] = options[0]
        used.add(options[0])

unmatched = transactions[is_return & ~transactions.index.isin(list(matched_sale))]
if len(unmatched):
    display(unmatched)
check(unmatched.empty, f"every return matches a sale ({int(is_return.sum())} returns)")

transactions["is_return"] = is_return
transactions["is_returned_sale"] = transactions.index.isin(list(matched_sale.values()))
transactions["matched_transaction_id"] = ""
for r_idx, s_idx in matched_sale.items():
    transactions.at[r_idx, "matched_transaction_id"] = transactions.at[s_idx, "transaction_id"]
    transactions.at[s_idx, "matched_transaction_id"] = transactions.at[r_idx, "transaction_id"]
transactions["exclude_from_metrics"] = transactions["is_return"] | transactions["is_returned_sale"]

pairs = pd.DataFrame([(transactions.at[r, "transaction_id"], transactions.at[s, "transaction_id"],
                       -transactions.at[r, "quantity"], transactions.at[s, "quantity"])
                      for r, s in matched_sale.items()],
                     columns=["return_id", "sale_id", "returned_qty", "sold_qty"])
display(pairs)
partial = pairs[pairs["returned_qty"] < pairs["sold_qty"]]
info(f"{len(partial)} partial returns (returned qty < sold qty); the whole sale is still flagged")
blank_cust = int((transactions.loc[list(matched_sale), "customer_id"].str.strip() == "").sum())
info(f"{blank_cust} returns were matched on a blank customer_id")

record("Negative values are returns matching earlier sales", "transactions",
       f"{len(matched_sale)} returns + {len(matched_sale)} matched sales",
       "Flagged both (`is_return`, `is_returned_sale`, `exclude_from_metrics`); kept in the data",
       "So they aren't counted in revenue or other metrics, but aren't dropped, so we can still "
       "track them as needed.")

PASS  negative total_amount occurs on exactly the rows with negative quantity


PASS  every return matches a sale (30 returns)


,return_id,sale_id,returned_qty,sold_qty
0,TXN20001,TXN10066,2,2
1,TXN20002,TXN10067,2,2
2,TXN20003,TXN10068,3,3
3,TXN20004,TXN10069,5,5
4,TXN20005,TXN10070,2,2
5,TXN20006,TXN10071,5,5
6,TXN20007,TXN10072,4,4
7,TXN20008,TXN10073,2,2
8,TXN20009,TXN10074,1,1
9,TXN20010,TXN10075,5,5


INFO  0 partial returns (returned qty < sold qty); the whole sale is still flagged
INFO  0 returns were matched on a blank customer_id


## 5. Stores that appear in transactions but not in the stores file

Some transactions belong to stores that aren't in the stores file at all, not even in the raw data. A placeholder row is added to `stores` for each of these stores so their sales stay in the data. The placeholder rows have:
- `store_name`, `city`, `state` and `region` set to `Unknown`
- `zip_code` and `opened_date` left blank
- `store_details_missing = True`

Every other store gets `store_details_missing = False`. The missing stores are saved to `stores_missing_from_source.csv` for the client.

This step runs after the transactions are cleaned, so it only adds stores that the validated transactions actually use.

In [16]:

missing_ids = sorted(set(transactions["store_id"]) - set(stores["store_id"]))
missing_txn = transactions[transactions["store_id"].isin(missing_ids)]
missing_stores = (missing_txn.groupby("store_id")
                  .agg(transaction_count=("transaction_id", "size"),
                       first_date=("transaction_date", "min"), last_date=("transaction_date", "max"))
                  .reset_index())
display(missing_stores)
expect("stores_missing_from_file", len(missing_ids), "store_ids in transactions that are not in the stores file")
info(f"{len(missing_txn)} transactions belong to these stores")

placeholders = pd.DataFrame({
    "store_id": missing_ids,
    "store_name": "Unknown", "city": "Unknown", "state": "Unknown",
    "zip_code": pd.Series([pd.NA] * len(missing_ids), dtype="string"),
    "region": "Unknown", "opened_date": "", "zip_code_invalid": False,
})
stores["store_details_missing"] = False
placeholders["store_details_missing"] = True
stores = pd.concat([stores, placeholders[stores.columns]], ignore_index=True)

check(transactions["store_id"].isin(stores["store_id"]).all(), "every transaction's store_id is now in stores")
check(stores["store_id"].is_unique, "store_id is unique")
record("Stores in transactions but not in the stores file", "stores",
       f"{len(missing_ids)} stores ({len(missing_txn)} transactions)",
       "Added placeholder stores (details `Unknown`, `store_details_missing` = True); sales kept; "
       "saved to `stores_missing_from_source.csv` for the client",
       "— (no rationale given)")

,store_id,transaction_count,first_date,last_date
0,S016,2,2026-05-02,2026-05-04
1,S017,1,2026-04-17,2026-04-17
2,S018,1,2026-05-06,2026-05-06
3,S019,1,2026-04-29,2026-04-29


PASS  store_ids in transactions that are not in the stores file: expected 4, found 4
INFO  5 transactions belong to these stores
PASS  every transaction's store_id is now in stores
PASS  store_id is unique


In [17]:
# Products that appear in transactions but not in the products file (P031, P032)
EXPECTED.setdefault("products_missing_from_file", 2)

# Rerun-safe: remove placeholders added by an earlier run of this cell
if "product_details_missing" in products.columns:
    products = products[~products["product_details_missing"]].drop(columns="product_details_missing")

missing_pids = sorted(set(transactions["product_id"]) - set(products["product_id"]))
missing_ptxn = transactions[transactions["product_id"].isin(missing_pids)]
missing_products = (missing_ptxn.groupby("product_id")
                    .agg(transaction_count=("transaction_id", "size"),
                         unit_prices=("unit_price", lambda p: ", ".join(f"{v:.2f}" for v in sorted(p.unique()))),
                         first_date=("transaction_date", "min"), last_date=("transaction_date", "max"))
                    .reset_index())
display(missing_products)
expect("products_missing_from_file", len(missing_pids), "product_ids in transactions that are not in the products file")
info(f"{len(missing_ptxn)} transactions belong to these products")

placeholder_products = pd.DataFrame({
    "product_id": missing_pids,
    "product_name": ["Product " + p for p in missing_pids],  # same pattern as every other product_name
    "category": "Unknown",
    "unit_price": float("nan"),                              # catalog price left blank
    "supplier_id": "Unknown",
})
products["product_details_missing"] = False
placeholder_products["product_details_missing"] = True
products = pd.concat([products, placeholder_products[products.columns]], ignore_index=True)

check(transactions["product_id"].isin(products["product_id"]).all(), "every transaction's product_id is now in products")
check(products["product_id"].is_unique, "product_id is unique")
record("Products in transactions but not in the products file", "products",
       f"{len(missing_pids)} products ({len(missing_ptxn)} transactions)",
       "Added placeholder products (category and supplier `Unknown`, blank catalog price, "
       "`product_details_missing` = True); sales kept; saved to `products_missing_from_source.csv` for the client",
       "— (no rationale given)")

,product_id,transaction_count,unit_prices,first_date,last_date
0,P031,2,"123.17, 157.39",2026-05-14,2026-05-19
1,P032,1,242.36,2026-05-02,2026-05-02


PASS  product_ids in transactions that are not in the products file: expected 2, found 2
INFO  3 transactions belong to these products
PASS  every transaction's product_id is now in products
PASS  product_id is unique


## 5. Final checks

These checks only print information. They show how the tables fit together after cleaning, but nothing here changes the data.

In [18]:
check(stores["store_id"].is_unique, "stores.store_id is unique")
check(products["product_id"].is_unique, "products.product_id is unique")

orphan_stores = ~transactions["store_id"].isin(stores["store_id"])
orphan_products = ~transactions["product_id"].isin(products["product_id"])
info(f"{int(orphan_stores.sum())} transactions with a store_id not in stores")
info(f"{int(orphan_products.sum())} transactions with a product_id not in products")

for name, df in {"stores": stores, "products": products, "transactions": transactions}.items():
    blanks = {c: int(is_blank(df[c]).sum()) for c in df.columns if df[c].dtype == object or str(df[c].dtype) == "string"}
    blanks = {c: n for c, n in blanks.items() if n}
    info(f"{name}: {len(df):,} rows; blank values: {blanks or 'none'}")

display(transactions.dtypes.to_frame("dtype").T)

PASS  stores.store_id is unique
PASS  products.product_id is unique
INFO  0 transactions with a store_id not in stores
INFO  0 transactions with a product_id not in products
INFO  stores: 19 rows; blank values: {'zip_code': 4, 'opened_date': 4}
INFO  products: 32 rows; blank values: none
INFO  transactions: 485 rows; blank values: {'customer_id': 40, 'matched_transaction_id': 425}


,transaction_id,transaction_date,store_id,product_id,customer_id,quantity,unit_price,total_amount,is_return,is_returned_sale,matched_transaction_id,exclude_from_metrics
dtype,object,datetime64[ns],object,object,object,int64,float64,float64,bool,bool,object,bool


## 6. Save the validated tables
`transaction_date` is written as `YYYY-MM-DD`, and money columns are written with 2 decimal places. `zip_code` is written as text. To keep its leading zeros when you read it back, use `pd.read_csv(..., dtype={"zip_code": str})`.

In [19]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
outputs = {
    "stores.csv": stores,
    "products.csv": products,
    "transactions.csv": transactions,
    "transactions_dropped_zero_quantity.csv": zero_dropped,
    "products_missing_from_source.csv": missing_products
}
for fname, df in outputs.items():
    df.to_csv(OUT_DIR / fname, index=False, date_format="%Y-%m-%d", float_format="%.2f")
    print(f"wrote {OUT_DIR / fname}  ({len(df):,} rows)")

log = pd.DataFrame(LOG)
log.to_csv(OUT_DIR / "cleaning_log.csv", index=False)
print(f"wrote {OUT_DIR / 'cleaning_log.csv'}")

wrote ..\data\validated\stores.csv  (19 rows)
wrote ..\data\validated\products.csv  (32 rows)
wrote ..\data\validated\transactions.csv  (485 rows)
wrote ..\data\validated\transactions_dropped_zero_quantity.csv  (5 rows)
wrote ..\data\validated\products_missing_from_source.csv  (2 rows)
wrote ..\data\validated\cleaning_log.csv


## 7. Decision log (counts from this run)

In [20]:
display(Markdown(to_markdown(log)))

| Issue | File | Count | Decision | Rationale |
|---|---|---|---|---|
| `region` blank for both stores with state 'OR' | stores | 2 | Filled `region` with "West" | Based on the other state/region columns, we can impute the region "West" here. |
| `S007` appears twice with different store_names | stores | 2 rows (1 dropped) | Dropped one row (kept the first in file order) | It's the same record. |
| Invalid zip codes | stores | 1 | Kept `zip_code` as text; flagged zips that are not 5 digits in `zip_code_invalid`; not imputed | Text avoids truncating leading/trailing zeros. It's too risky to impute new zips without being sure, but if zip code isn't necessary for analytics, it can be safe to keep in the data for now. |
| Products with no `category` | products | 5 | Filled from the category of other products with the same `supplier_id` | Each supplier maps to exactly one category. |
| Exact duplicate rows | products | 1 | Dropped | — (no rationale given) |
| `P005` appears twice with different prices | products | 2 rows (1 dropped) | Kept the row whose price matches the transactions `unit_price` | Only one matches the price in the transactions table. |
| `P027` has a catalog price of 0.00 | products | 1 | Imputed the price from the transactions `unit_price` (195.34) | It has a price in the transactions table. |
| Exact duplicate rows | transactions | 15 | Deduplicated | — (no rationale given) |
| `$` in `total_amount` | transactions | 25 | Removed `$`, cast `total_amount` to a decimal number | — (no rationale given) |
| `quantity` and `total_amount` both 0 | transactions | 5 | Dropped; saved to `transactions_dropped_zero_quantity.csv` for the client | We drop them so they don't skew average transaction value or revenue tracking. |
| Mixed date formats; day/month ambiguous where both <= 12 | transactions | 485 dates cast; 7 rows (7 values) resolved by decision | Cast to YYYY-MM-DD; ambiguous values resolved as listed in the config | The dates range from March to June: 01-04-2026 (TXN10011) is April 1st; 04/12/2026 and 04/01/2026 are April; 05/07/2026 is May; 12-03-2026 is March; 05/05/2026 can go either way, so it doesn't matter. That leaves one record, 06-04-2026, which is June 4th, 2026. |
| Negative values are returns matching earlier sales | transactions | 30 returns + 30 matched sales | Flagged both (`is_return`, `is_returned_sale`, `exclude_from_metrics`); kept in the data | So they aren't counted in revenue or other metrics, but aren't dropped, so we can still track them as needed. |
| Stores in transactions but not in the stores file | stores | 4 stores (5 transactions) | Added placeholder stores (details `Unknown`, `store_details_missing` = True); sales kept; saved to `stores_missing_from_source.csv` for the client | — (no rationale given) |
| Products in transactions but not in the products file | products | 2 products (3 transactions) | Added placeholder products (category and supplier `Unknown`, blank catalog price, `product_details_missing` = True); sales kept; saved to `products_missing_from_source.csv` for the client | — (no rationale given) |